In [62]:
import argparse
import json
import tensorboard
import tensorboardX
import os
import re
import argparse
import json
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim 
import nni
from nni.nas.nn.pytorch import ModelSpace, LayerChoice, MutableConv2d, MutableBatchNorm2d, MutableReLU, MutableLinear
from nni.nas.experiment.config import NasExperimentConfig
from pytorch_lightning import Trainer
from nni.nas.evaluator.pytorch import Lightning, ClassificationModule, Trainer
from nni.nas.experiment import NasExperiment
from nni.nas.space import model_context
from nni.nas.hub.pytorch import DARTS
from nni.nas.strategy import DARTS as DartsStrategy
from pytorch_lightning.loggers import TensorBoardLogger
from torch.utils.data import DataLoader
from torch.utils.data.sampler import SubsetRandomSampler
from torchvision import transforms
from torchvision.datasets import CIFAR10
from nni.nas.experiment import NasExperiment
from nni.nas.evaluator import FunctionalEvaluator
import nni.nas.strategy as strategy
from torchvision import transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from nni.experiment.config import utils, ExperimentConfig
from pytorch_lightning.callbacks import ModelCheckpoint
torch.set_float32_matmul_precision('medium')
from tqdm import tqdm
from nni.nas.nn.pytorch import LayerChoice, ModelSpace, ValueChoice
from torch.utils.data import DataLoader, Dataset, SubsetRandomSampler
from pytorch_lightning import LightningModule, Trainer
from torchvision import datasets, transforms
from nni.nas.evaluator.pytorch import Classification

# Import the quantized CustomDARTSSpace
import sys
import importlib
import DoReFaLayers
importlib.reload(DoReFaLayers)
from DoReFaLayers import MutableDoReFaConv2d, MutableDoReFaLinear
from nni.nas.oneshot.pytorch.supermodule.operation import MixedConv2d, MixedLinear


class SafeMixedConv2d(MixedConv2d):
    def freeze(self, sample):
        arguments = {**self.init_arguments}
        for name, mutable in self.mutable_arguments.items():
            arguments[name] = mutable.freeze(sample)
        operation = self.bound_type(**arguments)
        state_dict = self.freeze_weight(**arguments)
        operation.load_state_dict(state_dict, strict=False)
        return operation


class SafeMixedLinear(MixedLinear):
    def freeze(self, sample):
        arguments = {**self.init_arguments}
        for name, mutable in self.mutable_arguments.items():
            arguments[name] = mutable.freeze(sample)
        operation = self.bound_type(**arguments)
        state_dict = self.freeze_weight(**arguments)
        operation.load_state_dict(state_dict, strict=False)
        return operation


def safe_mutable_conv_hook(module, name, memo, kwargs):
    if isinstance(module, MutableDoReFaConv2d):
        return SafeMixedConv2d.mutate(module, name, memo, kwargs)
    return False


def safe_mutable_linear_hook(module, name, memo, kwargs):
    if isinstance(module, MutableDoReFaLinear):
        return SafeMixedLinear.mutate(module, name, memo, kwargs)
    return False

# Quantized DARTS Architecture Search

This notebook performs NAS using the quantized CustomDARTSSpace with DoReFa quantization-aware training.

# Data Loading and Augmentation

In [34]:
def cutout_transform(img, length: int = 16):
    h, w = img.size(1), img.size(2)
    mask = np.ones((h, w), np.float32)
    y = np.random.randint(h)
    x = np.random.randint(w)

    y1 = np.clip(y - length // 2, 0, h)
    y2 = np.clip(y + length // 2, 0, h)
    x1 = np.clip(x - length // 2, 0, w)
    x2 = np.clip(x + length // 2, 0, w)

    mask[y1: y2, x1: x2] = 0.
    mask = torch.from_numpy(mask)
    mask = mask.expand_as(img)
    img *= mask
    return img

In [35]:
def get_cifar10_dataset(train: bool = True, cutout: bool = False):
    if train:
        transform_list = [
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ToTensor(), 
        ]
        if cutout:
            transform_list.append(cutout_transform)
        transform = transforms.Compose(transform_list)
    else:
        transform = transforms.Compose([
            transforms.ToTensor(), 
        ])

    dataset = nni.trace(CIFAR10)(root='./data', train=train, download=True, transform=transform)
    
    return dataset

batch_size = 128
train_data = get_cifar10_dataset()
test_data = get_cifar10_dataset(train=False)

train_loader = DataLoader(
    train_data, batch_size=batch_size,
    pin_memory=True, num_workers=6, persistent_workers=True, shuffle=True
)

valid_loader = DataLoader(
    test_data, batch_size=batch_size,
    pin_memory=True, num_workers=6, persistent_workers=True
)

# Classification Module

In [36]:
@nni.trace
class DartsClassificationModule(ClassificationModule):
    def __init__(self, learning_rate: float = 0.001, weight_decay: float = 0., auxiliary_loss_weight: float = 0.4, max_epochs: int = 600):
        super().__init__(learning_rate=learning_rate, weight_decay=weight_decay, export_onnx=False, num_classes=10)        
        self.auxiliary_loss_weight = auxiliary_loss_weight
        self.max_epochs = max_epochs
        self.learning_rate = learning_rate

    def configure_optimizers(self):
        optimizer = torch.optim.SGD(self.parameters(), lr=self.learning_rate, momentum=0.9, weight_decay=0.)
        return {
            'optimizer': optimizer,
            'lr_scheduler': torch.optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)        }

    def training_step(self, batch, batch_idx):
        """Training step, customized with auxiliary loss."""
        x, y = batch
        if self.auxiliary_loss_weight:
            y_hat, y_aux = self(x)
            loss_main = self.criterion(y_hat, y)
            loss_aux = self.criterion(y_aux, y)
            self.log('train_loss_main', loss_main)
            self.log('train_loss_aux', loss_aux)
            loss = loss_main + self.auxiliary_loss_weight * loss_aux
        else:
            y_hat = self(x)
            loss = self.criterion(y_hat, y)
        self.log('train_loss', loss, prog_bar=True)
        for name, metric in self.metrics.items():
            self.log('train_' + name, metric(y_hat, y), prog_bar=True)
        return loss

    def on_train_epoch_start(self):
        # Logging learning rate at the beginning of every epoch
        self.log('lr', self.trainer.optimizers[0].param_groups[0]['lr'])

In [37]:
class CustomDARTSSpace(ModelSpace):
	"""Quantization-aware DARTS search space with DoReFa quantization.

	This corresponds to a flexible search space with:
	- LayerChoice on first 2 blocks (pool/conv ordering)
	- channel choices for layers 1-6
	- all learnable conv/linear ops quantized via DoReFa wrappers
	"""

	def __init__(
		self,
		input_channels: int = 3,
		channels: int = 64,
		num_classes: int = 10,
		layers: int = 7,
		verbose: int = 0,
		drop_path_prob: float = 0.1,
		num_bits: int = 4,
	):
		super().__init__()
		self.layers = nn.ModuleList()
		self.drop_path_prob = drop_path_prob
		self.verbose = verbose

		layer0_out = 16
		layer1_out = nni.choice("layer1_out_channels", [16, 32, 64])
		layer2_out = nni.choice("layer2_out_channels", [16, 32, 64])
		layer3_out = nni.choice("layer3_out_channels", [16, 32, 64])
		layer4_out = nni.choice("layer4_out_channels", [16, 32, 64])
		layer5_out = nni.choice("layer5_out_channels", [16, 32, 64])
		layer6_out = nni.choice("layer6_out_channels", [16, 32, 64])
		layer7_out = 22

		# Quantize first stem conv as well for full QAT consistency.
		self.preliminary_layer = MutableDoReFaConv2d(
			input_channels,
			layer0_out,
			kernel_size=3,
			padding=0,
			bias=False,
			num_bits=num_bits,
		)
		self.layer0_bn = MutableBatchNorm2d(layer0_out)
		self.layer0_relu = MutableReLU()

		layer1 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer0_out,
						layer1_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer1_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer0_out,
						layer1_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer1_out),
					MutableReLU(),
				),
			],
			label="layer_1",
		)
		self.layers.append(layer1)

		layer2 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer1_out,
						layer2_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer2_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer1_out,
						layer2_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer2_out),
					MutableReLU(),
				),
			],
			label="layer_2",
		)
		self.layers.append(layer2)

		layer3 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer2_out,
						layer3_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer3_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer2_out,
						layer3_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer3_out),
					MutableReLU(),
				),
			],
			label="layer_3",
		)
		self.layers.append(layer3)

		layer4 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer3_out,
						layer4_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer4_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer3_out,
						layer4_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer4_out),
					MutableReLU(),
				),
			],
			label="layer_4",
		)
		self.layers.append(layer4)

		layer5 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer4_out,
						layer5_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer5_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer4_out,
						layer5_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer5_out),
					MutableReLU(),
				),
			],
			label="layer_5",
		)
		self.layers.append(layer5)

		layer6 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer5_out,
						layer6_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer6_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer5_out,
						layer6_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer6_out),
					MutableReLU(),
				),
			],
			label="layer_6",
		)
		self.layers.append(layer6)

		layer7 = LayerChoice(
			[
				nn.Sequential(
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableDoReFaConv2d(
						layer6_out,
						layer7_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					MutableBatchNorm2d(layer7_out),
					MutableReLU(),
				),
				nn.Sequential(
					MutableDoReFaConv2d(
						layer6_out,
						layer7_out,
						kernel_size=3,
						bias=False,
						num_bits=num_bits,
					),
					nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
					MutableBatchNorm2d(layer7_out),
					MutableReLU(),
				),
			],
			label="layer_7",
		)
		self.layers.append(layer7)

		self.pool = nn.AdaptiveAvgPool2d((3, 3))
		feature1 = nni.choice("feature1", [32, 64, 128])
		feature2 = nni.choice("feature2", [32, 64, 128])
		feature3 = nni.choice("feature3", [32, 64])
		self.fc1 = MutableDoReFaLinear(198, feature1, num_bits=num_bits)
		self.fc2 = MutableDoReFaLinear(feature1, feature2, num_bits=num_bits)
		self.fc3 = MutableDoReFaLinear(feature2, feature3, num_bits=num_bits)
		self.relu = nn.ReLU()
		self.classifier = MutableDoReFaLinear(feature3, num_classes, num_bits=num_bits)

	def forward(self, x: torch.Tensor) -> torch.Tensor:
		x = self.preliminary_layer(x)
		x = self.layer0_bn(x)
		x = self.layer0_relu(x)
		if self.verbose == 1:
			print(f"After preliminary layer: {x.shape}")

		for i, layer in enumerate(self.layers):
			x = layer(x)
			if self.verbose == 1:
				print(f"After layer {i + 1}: {x.shape}")
			if i in (1, 3, 6):
				x = nn.AvgPool2d(kernel_size=2, stride=2)(x)
				if self.verbose == 1:
					print(f"After avg pooling: {x.shape}")

		x = self.pool(x)
		if self.verbose == 1:
			print(f"After adaptive pooling: {x.shape}")

		x = torch.flatten(x, 1)
		if self.verbose == 1:
			print(f"After flattening: {x.shape}")

		x = self.fc1(x)
		x = self.relu(x)
		if self.verbose == 1:
			print(f"After fc1: {x.shape}")

		x = self.fc2(x)
		x = self.relu(x)
		if self.verbose == 1:
			print(f"After fc2: {x.shape}")

		x = self.fc3(x)
		x = self.relu(x)
		if self.verbose == 1:
			print(f"After fc3: {x.shape}")

		x = self.classifier(x)
		if self.verbose == 1:
			print(f"After classifier: {x.shape}")

		return x

	def set_drop_path_prob(self, drop_path_prob: float):
		self.drop_path_prob = drop_path_prob
		for layer in self.layers:
			if hasattr(layer, "set_drop_path_prob"):
				layer.set_drop_path_prob(drop_path_prob)

# Evaluator Setup

Checkpoint callback and Lightning Evaluator configuration

In [38]:
# Checkpoint callback
checkpoint_callback = ModelCheckpoint(
    monitor='train_acc', 
    dirpath='./checkpoints',
    filename='best-checkpoint',
    save_top_k=1,
    mode='max'
)

In [39]:
from nni.nas.evaluator.pytorch import Lightning, Trainer

max_epochs = 200

evaluator = Lightning(
    DartsClassificationModule(1e-2, 0., 0., max_epochs),
    Trainer(
        accelerator="auto",
        callbacks=[checkpoint_callback],
        max_epochs=max_epochs
    ),
    train_dataloaders=train_loader,
    val_dataloaders=valid_loader
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/nni/nas/evaluator/pytorch/lightning.py:139: When using training service to spawn trials, please try to wrap PyTorch DataLoader with nni.trace or import DataLoader from nni.nas.evaluator.pytorch.lightning: <torch.utils.data.dataloader.DataLoader object at 0x7f1933619a10>
/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/nni/nas/evaluator/pytorch/lightning.py:143: When using training service to spawn trials, please try to wrap PyTorch DataLoader with nni.trace or import DataLoader from nni.nas.evaluator.pytorch.lightning: <torch.utils.data.dataloader.DataLoader object at 0x7f19332ddf10>


# NAS Search with Quantized Model

Using CustomDARTSSpace with DoReFa quantization

In [40]:
def get_next_experiment_name(experiment_working_directory: str, base_name: str = "PhotonicDARTS_Quant-v"):
    """Generate the next experiment name by incrementing the version number."""
    if not os.path.exists(experiment_working_directory):
        os.makedirs(experiment_working_directory)
    
    dirs = os.listdir(experiment_working_directory)
    pattern = re.compile(rf'{base_name}(\d+)')

    highest_i = 0
    for d in dirs:
        match = pattern.match(d)
        if match:
            i_value = int(match.group(1))
            if i_value > highest_i:
                highest_i = i_value
    return f"{base_name}{highest_i + 1}"

In [ ]:
strategy = DartsStrategy(gradient_clip_val=0., mutation_hooks=[safe_mutable_conv_hook, safe_mutable_linear_hook])

def search(log_dir: str, batch_size: int = 128, num_bits: int = 4):
    """
    Run DARTS architecture search with quantized model.
    
    Parameters:
    -----------
    log_dir : str
        Directory for logging
    batch_size : int
        Batch size for training
    num_bits : int
        Number of bits for DoReFa quantization (default: 4)
    
    Returns:
    --------
    NasExperiment
        The completed NAS experiment with search results
    """
    
    # Define model search space with quantization
    model_space = CustomDARTSSpace(
        input_channels=3, 
        channels=64, 
        num_classes=10, 
        layers=7, 
        verbose=0,
        num_bits=num_bits  # Quantization parameter
    )
    model_space.set_drop_path_prob(0.)

    # Run NAS experiment
    exp_config = NasExperimentConfig.default(model_space, evaluator, strategy)
    exp_config.experiment_working_directory = "./DartsCheckpoints_Quant"
    exp_config.experiment_name = get_next_experiment_name("./DartsCheckpoints_Quant")
    exp_config.trial_concurrency = 1
    
    experiment = NasExperiment(model_space, evaluator, strategy, config=exp_config)
    experiment.run()

    return experiment

In [42]:
# Run the quantized DARTS search
# Set num_bits to 4 for 4-bit quantization (can adjust as needed)
experiment_results = search("./", 32, num_bits=4)

[2026-04-29 17:52:34] Config is not provided. Will try to infer.
[2026-04-29 17:52:34] Strategy is found to be a one-shot strategy. Setting execution engine to "sequential" and format to "raw".
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.


[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.


/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/nni/nas/evaluator/pytorch/lightning.py:139: When using training service to spawn trials, please try to wrap PyTorch DataLoader with nni.trace or import DataLoader from nni.nas.evaluator.pytorch.lightning: {'train': <torch.utils.data.dataloader.DataLoader object at 0x7f1933619a10>, 'val': <torch.utils.data.dataloader.DataLoader object at 0x7f19332ddf10>}


[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: Checkpoint callback does not have last_model_path or best_model_path attribute. Either the strategy has not started, or it did not save any checkpoint: <pytorch_lightning.callbacks.model_checkpoint.ModelCheckpoint object at 0x7f19333d9250>
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] WARNING: `training_service` will be ignored for sequential execution engine.
[2026-04-29 17:52:34] Checkpoint sav

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type                      | Params | Mode  | FLOPs
------------------------------------------------------------------------------
0 | training_module | DartsClassificationModule | 465 K  | train | 0    
------------------------------------------------------------------------------
465 K     Trainable params
0         Non-trainable params
465 K     Total params
1.862     Total estimated model params size (MB)
167       Modules in train mode
0         Modules in eval mode
0         Total Flops


Epoch 199: 100%|██████████████████| 391/391 [00:58<00:00,  6.64it/s, v_num=0, train_loss=0.254, train_acc=0.925]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 199: 100%|██████████████████| 391/391 [00:58<00:00,  6.64it/s, v_num=0, train_loss=0.254, train_acc=0.925]
[2026-04-29 21:10:26] Waiting for models submitted to engine to finish...
[2026-04-29 21:10:26] Experiment is completed.
[2026-04-29 21:10:26] WARNING: `training_service` will be ignored for sequential execution engine.


# Results and Best Architecture

In [48]:
# Export best architecture
best_arch = experiment_results.export_top_models(formatter='instance', top_k=1)[0]
best_arch_desc = experiment_results.export_top_models(formatter='dict', top_k=1)[0]
best_arch_state_dict = best_arch.state_dict()

print("Best Architecture Description:")
print(best_arch_desc)

Best Architecture Description:
{'layer_1': 1, 'layer1_out_channels': 16, 'layer_2': 1, 'layer2_out_channels': 16, 'layer_3': 1, 'layer3_out_channels': 64, 'layer_4': 1, 'layer4_out_channels': 32, 'layer_5': 0, 'layer5_out_channels': 64, 'layer_6': 1, 'layer6_out_channels': 16, 'layer_7': 1, 'feature1': 32, 'feature2': 32, 'feature3': 32}


In [49]:
# Experiment configuration
print("Experiment Configuration:")
print(experiment_results.config)

Experiment Configuration:
NasExperimentConfig(experiment_name='PhotonicDARTS_Quant-v1', experiment_type='nas', search_space_file=None, search_space='_reserved_', trial_command='', trial_code_directory='.', trial_concurrency=1, trial_gpu_number=None, max_experiment_duration=None, max_trial_number=None, max_trial_duration=None, nni_manager_ip=None, use_annotation=False, debug=False, log_level=None, experiment_working_directory='./DartsCheckpoints_Quant', tuner_gpu_indices=None, tuner=_AlgorithmConfig(name='_none_', class_name=None, code_directory=None, class_args={}), assessor=_AlgorithmConfig(name='_none_', class_name=None, code_directory=None, class_args={}), advisor=_AlgorithmConfig(name='_none_', class_name=None, code_directory=None, class_args={}), training_service=LocalConfig(platform='local', trial_command=<dataclasses._MISSING_TYPE object at 0x7f1b8594f950>, trial_code_directory=<dataclasses._MISSING_TYPE object at 0x7f1b8594f950>, trial_gpu_number=<dataclasses._MISSING_TYPE obje

# Auto arch code generator

### Load Checkpoint

In [51]:
checkpoint_path = './checkpoints/best-checkpoint.ckpt'

checkpoint = torch.load(checkpoint_path, map_location=torch.device('cpu'))

checkpoint_model = CustomDARTSSpace(input_channels=3, channels=64, num_classes=10, layers=7, verbose=0, num_bits=4)

checkpoint_state_dict = checkpoint_model.state_dict()
pretrained_state_dict = checkpoint['state_dict']

if 'global_step' in checkpoint:
    print('Global Step:', checkpoint['global_step'])

if 'callbacks' in checkpoint and isinstance(checkpoint['callbacks'], dict):
    for key, callback in checkpoint['callbacks'].items():
        if isinstance(callback, dict) and 'best_model_score' in callback:
            print('Best Model Score (Train Accuracy):', callback['best_model_score'].item())

Global Step: 150144
Best Model Score (Train Accuracy): 0.9750000238418579


In [85]:
# Load Lightning checkpoint into the training module and run a quick eval of the trained model
checkpoint_path = './checkpoints/best-checkpoint.ckpt'
print('Checkpoint path:', checkpoint_path)

# Try the standard Lightning loader first
try:
    trained_module = DartsClassificationModule.load_from_checkpoint(checkpoint_path, map_location='cpu')
    print('Successfully loaded module with load_from_checkpoint')
except Exception as e:
    print('load_from_checkpoint failed:', e)
    # Fallback: construct a fresh module and manually load state_dict (strip Lightning prefixes)
    trained_module = DartsClassificationModule(1e-2, 0., 0., max_epochs)
    ckpt = __import__('torch').load(checkpoint_path, map_location='cpu')
    sd = ckpt.get('state_dict', {})
    stripped = {}
    for k, v in sd.items():
        nk = k
        if nk.startswith('training_module._model.'):
            nk = nk.replace('training_module._model.', '')
        elif nk.startswith('training_module.'):
            nk = nk.replace('training_module.', '')
        stripped[nk] = v
    rep = trained_module.load_state_dict(stripped, strict=False)
    print('Manual load_state_dict report:', rep)

# Ensure the wrapper has the model instance attached (use `scratch_model` already built above)
try:
    trained_module.set_model(scratch_model)
    print('Attached `scratch_model` to trained_module via set_model()')
except Exception:
    pass

# Extract the actual model to evaluate (training wrapper may store it in `_model`)
if hasattr(trained_module, '_model'):
    model_to_eval = trained_module._model
else:
    model_to_eval = trained_module

# Quick evaluation: first 10 validation batches
model_to_eval.to(device)
model_to_eval.eval()
criterion = __import__('torch').nn.CrossEntropyLoss()
total = 0
correct = 0
loss_sum = 0.0
with __import__('torch').no_grad():
    for i, (inputs, labels) in enumerate(valid_loader):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model_to_eval(inputs)
        loss = criterion(outputs, labels)
        loss_sum += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        if i >= 9:
            break
avg_loss = loss_sum / (i + 1) if (i + 1) > 0 else float('nan')
acc = correct / total if total > 0 else 0.0
print(f'Checkpoint quick eval on {total} samples: loss={avg_loss:.4f}, acc={acc:.4%}')
print('Sample predictions vs labels:')
for inputs, labels in valid_loader:
    inputs, labels = inputs.to(device), labels.to(device)
    outs = model_to_eval(inputs)
    print('preds', outs.argmax(dim=1)[:10].cpu().tolist())
    print('true ', labels[:10].cpu().tolist())
    break

# Re-export best_arch variables for downstream cells
best_arch = experiment_results.export_top_models(formatter='instance', top_k=1)[0]
best_arch_desc = experiment_results.export_top_models(formatter='dict', top_k=1)[0]
best_arch_state_dict = best_arch.state_dict()


Checkpoint path: ./checkpoints/best-checkpoint.ckpt
load_from_checkpoint failed: Error(s) in loading state_dict for DartsClassificationModule:
	Unexpected key(s) in state_dict: "training_module._model.layers.0._arch_alpha", "training_module._model.layers.0.0.1.weight", "training_module._model.layers.0.0.1._arch_alpha.layer1_out_channels", "training_module._model.layers.0.0.2.weight", "training_module._model.layers.0.0.2.bias", "training_module._model.layers.0.0.2.running_mean", "training_module._model.layers.0.0.2.running_var", "training_module._model.layers.0.0.2.num_batches_tracked", "training_module._model.layers.0.0.2._arch_alpha.layer1_out_channels", "training_module._model.layers.0.1.0.weight", "training_module._model.layers.0.1.0._arch_alpha.layer1_out_channels", "training_module._model.layers.0.1.2.weight", "training_module._model.layers.0.1.2.bias", "training_module._model.layers.0.1.2.running_mean", "training_module._model.layers.0.1.2.running_var", "training_module._model.la

Manual load_state_dict report: _IncompatibleKeys(missing_keys=[], unexpected_keys=['layers.0._arch_alpha', 'layers.0.0.1.weight', 'layers.0.0.1._arch_alpha.layer1_out_channels', 'layers.0.0.2.weight', 'layers.0.0.2.bias', 'layers.0.0.2.running_mean', 'layers.0.0.2.running_var', 'layers.0.0.2.num_batches_tracked', 'layers.0.0.2._arch_alpha.layer1_out_channels', 'layers.0.1.0.weight', 'layers.0.1.0._arch_alpha.layer1_out_channels', 'layers.0.1.2.weight', 'layers.0.1.2.bias', 'layers.0.1.2.running_mean', 'layers.0.1.2.running_var', 'layers.0.1.2.num_batches_tracked', 'layers.0.1.2._arch_alpha.layer1_out_channels', 'layers.1._arch_alpha', 'layers.1.0.1.weight', 'layers.1.0.1._arch_alpha.layer1_out_channels', 'layers.1.0.1._arch_alpha.layer2_out_channels', 'layers.1.0.2.weight', 'layers.1.0.2.bias', 'layers.1.0.2.running_mean', 'layers.1.0.2.running_var', 'layers.1.0.2.num_batches_tracked', 'layers.1.0.2._arch_alpha.layer2_out_channels', 'layers.1.1.0.weight', 'layers.1.1.0._arch_alpha.laye

# From-Scratch Model

In [54]:
class PhotonicArch(torch.nn.Module):
    def __init__(self, drop_path_prob=0.0, arch_dict=None):
        super().__init__()
        self.arch_dict = arch_dict
        self.drop_path_prob = drop_path_prob

        self.layer0_conv = torch.nn.Conv2d(3, 8, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = torch.nn.BatchNorm2d(8)
        self.layer0_relu = torch.nn.ReLU(inplace=False)

        if arch_dict['layer_1'] == 0:
            self.layer1_conv = torch.nn.Conv2d(8, arch_dict['layer1_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer1_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer1_bn = torch.nn.BatchNorm2d(arch_dict['layer1_out_channels'], affine=True)
            self.layer1_relu = torch.nn.ReLU(inplace=False)
        else:
            self.layer1_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer1_conv = torch.nn.Conv2d(8, arch_dict['layer1_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer1_bn = torch.nn.BatchNorm2d(arch_dict['layer1_out_channels'], affine=True)
            self.layer1_relu = torch.nn.ReLU(inplace=False)

        if arch_dict['layer_2'] == 0:
            self.layer2_conv = torch.nn.Conv2d(arch_dict['layer1_out_channels'], arch_dict['layer2_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer2_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer2_bn = torch.nn.BatchNorm2d(arch_dict['layer2_out_channels'], affine=True)
            self.layer2_relu = torch.nn.ReLU(inplace=False)
        else:
            self.layer2_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer2_conv = torch.nn.Conv2d(arch_dict['layer1_out_channels'], arch_dict['layer2_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer2_bn = torch.nn.BatchNorm2d(arch_dict['layer2_out_channels'], affine=True)
            self.layer2_relu = torch.nn.ReLU(inplace=False)

        if arch_dict['layer_3'] == 0:
            self.layer3_conv = torch.nn.Conv2d(arch_dict['layer2_out_channels'], arch_dict['layer3_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer3_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer3_bn = torch.nn.BatchNorm2d(arch_dict['layer3_out_channels'], affine=True)
            self.layer3_relu = torch.nn.ReLU(inplace=False)
        else:
            self.layer3_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer3_conv = torch.nn.Conv2d(arch_dict['layer2_out_channels'], arch_dict['layer3_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer3_bn = torch.nn.BatchNorm2d(arch_dict['layer3_out_channels'], affine=True)
            self.layer3_relu = torch.nn.ReLU(inplace=False)

        if arch_dict['layer_4'] == 0:
            self.layer4_conv = torch.nn.Conv2d(arch_dict['layer3_out_channels'], arch_dict['layer4_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer4_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer4_bn = torch.nn.BatchNorm2d(arch_dict['layer4_out_channels'], affine=True)
            self.layer4_relu = torch.nn.ReLU(inplace=False)
        else:
            self.layer4_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer4_conv = torch.nn.Conv2d(arch_dict['layer3_out_channels'], arch_dict['layer4_out_channels'], kernel_size=3, stride=1, padding=1)
            self.layer4_bn = torch.nn.BatchNorm2d(arch_dict['layer4_out_channels'], affine=True)
            self.layer4_relu = torch.nn.ReLU(inplace=False)

        if arch_dict['layer_5'] == 0:
            self.layer5_conv = torch.nn.Conv2d(arch_dict['layer4_out_channels'], 22, kernel_size=3, stride=1, padding=1)
            self.layer5_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer5_bn = torch.nn.BatchNorm2d(22, affine=True)
            self.layer5_relu = torch.nn.ReLU(inplace=False)
        else:
            self.layer5_avgpool = torch.nn.AvgPool2d(kernel_size=3, stride=1, padding=0)
            self.layer5_conv = torch.nn.Conv2d(arch_dict['layer4_out_channels'], 22, kernel_size=3, stride=1, padding=1)
            self.layer5_bn = torch.nn.BatchNorm2d(22, affine=True)
            self.layer5_relu = torch.nn.ReLU(inplace=False)

        self.pool = torch.nn.AdaptiveAvgPool2d((3, 3))
        self.fc1 = torch.nn.Linear(198, 160)
        self.fc2 = torch.nn.Linear(160, 128)
        self.fc3 = torch.nn.Linear(128, 96)
        self.relu = torch.nn.ReLU(inplace=False)
        self.classifier = torch.nn.Linear(96, 10)

    def forward(self, x):
        x = self.layer0_conv(x)
        x = self.layer0_bn(x)
        x = self.layer0_relu(x)

        x = self.layer1_avgpool(x)
        x = self.layer1_conv(x)
        x = self.layer1_bn(x)
        x = self.layer1_relu(x)

        x = self.layer2_conv(x)
        x = self.layer2_avgpool(x)
        x = self.layer2_bn(x)
        x = self.layer2_relu(x)

        x = self.layer3_conv(x)
        x = self.layer3_avgpool(x)
        x = self.layer3_bn(x)
        x = self.layer3_relu(x)

        x = torch.nn.AvgPool2d(kernel_size=2, stride=2)(x)

        x = self.layer4_avgpool(x)
        x = self.layer4_conv(x)
        x = self.layer4_bn(x)
        x = self.layer4_relu(x)

        x = self.layer5_avgpool(x)
        x = self.layer5_conv(x)
        x = self.layer5_bn(x)
        x = self.layer5_relu(x)

        x = torch.nn.AvgPool2d(kernel_size=2, stride=2)(x)
        x = self.pool(x)

        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.classifier(x)
        return x

    def set_drop_path_prob(self, drop_path_prob):
        self.drop_path_prob = drop_path_prob

In [67]:
scratch_model = best_arch.freeze(best_arch_desc)
scratch_model

CustomDARTSSpace(
  (layers): ModuleList(
    (0-1): 2 x Sequential(
      (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), bias=False)
      (1): AvgPool2d(kernel_size=2, stride=1, padding=1)
      (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): ReLU()
    )
    (2): Sequential(
      (0): Conv2d(16, 64, kernel_size=(3, 3), stride=(1, 1), bias=False)
      (1): AvgPool2d(kernel_size=2, stride=1, padding=1)
      (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): ReLU()
    )
    (3): Sequential(
      (0): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
      (1): AvgPool2d(kernel_size=2, stride=1, padding=1)
      (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): ReLU()
    )
    (4): Sequential(
      (0): AvgPool2d(kernel_size=2, stride=1, padding=1)
      (1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), bias=False)
   

In [74]:
import copy

def build_frozen_dorefa_model(model: nn.Module, num_bits: int = 4) -> nn.Module:
    model_copy = copy.deepcopy(model)

    def _replace_modules(module: nn.Module) -> None:
        for name, child in list(module.named_children()):
            if isinstance(child, nn.Conv2d):
                replacement = DoReFaLayers.FrozenDoReFaConv2d(
                    in_channels=child.in_channels,
                    out_channels=child.out_channels,
                    kernel_size=child.kernel_size,
                    stride=child.stride,
                    padding=child.padding,
                    dilation=child.dilation,
                    groups=child.groups,
                    bias=child.bias is not None,
                    padding_mode=child.padding_mode,
                    device=child.weight.device,
                    dtype=child.weight.dtype,
                    num_bits=num_bits,
                )
                replacement.load_state_dict(child.state_dict())
                setattr(module, name, replacement)
            elif isinstance(child, nn.Linear):
                replacement = DoReFaLayers.FrozenDoReFaLinear(
                    in_features=child.in_features,
                    out_features=child.out_features,
                    bias=child.bias is not None,
                    device=child.weight.device,
                    dtype=child.weight.dtype,
                    num_bits=num_bits,
                )
                replacement.load_state_dict(child.state_dict())
                setattr(module, name, replacement)
            else:
                _replace_modules(child)

    _replace_modules(model_copy)
    return model_copy

scratch_model_frozen_dorefa = build_frozen_dorefa_model(scratch_model, num_bits=4)
scratch_model_frozen_dorefa

CustomDARTSSpace(
  (layers): ModuleList(
    (0-1): 2 x Sequential(
      (0): FrozenDoReFaConv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), bias=False)
      (1): AvgPool2d(kernel_size=2, stride=1, padding=1)
      (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): ReLU()
    )
    (2): Sequential(
      (0): FrozenDoReFaConv2d(16, 64, kernel_size=(3, 3), stride=(1, 1), bias=False)
      (1): AvgPool2d(kernel_size=2, stride=1, padding=1)
      (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): ReLU()
    )
    (3): Sequential(
      (0): FrozenDoReFaConv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
      (1): AvgPool2d(kernel_size=2, stride=1, padding=1)
      (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): ReLU()
    )
    (4): Sequential(
      (0): AvgPool2d(kernel_size=2, stride=1, padding=1)
      (1): FrozenDoReFaConv2d(32, 64, ke

In [72]:
import json

# Export the best architecture description to JSON for use in other scripts
arch_export_path = 'best_arch_darts.json'
with open(arch_export_path, 'w') as f:
    json.dump(best_arch_desc, f, indent=2)
print(f"Architecture exported to {arch_export_path}")
print("Architecture choices:", best_arch_desc)

Architecture exported to best_arch_darts.json
Architecture choices: {'layer_1': 1, 'layer1_out_channels': 16, 'layer_2': 1, 'layer2_out_channels': 16, 'layer_3': 1, 'layer3_out_channels': 64, 'layer_4': 1, 'layer4_out_channels': 32, 'layer_5': 0, 'layer5_out_channels': 64, 'layer_6': 1, 'layer6_out_channels': 16, 'layer_7': 1, 'feature1': 32, 'feature2': 32, 'feature3': 32}


In [76]:
# Use the DoReFa-frozen variant for all downstream training/evaluation cells.
scratch_model = scratch_model_frozen_dorefa
print(type(scratch_model).__name__)

CustomDARTSSpace


### Rename dictionary keys

In [77]:
best_arch_state_dict = scratch_model.state_dict()

In [86]:
# Compare checkpoint parameter tensors with `scratch_model` state_dict and report mismatches
import torch
from collections import defaultdict

ckpt_path = './checkpoints/best-checkpoint.ckpt'
ckpt = torch.load(ckpt_path, map_location='cpu')
sd_ckpt = ckpt.get('state_dict', {})
# strip common lightning prefixes
sd_stripped = {}
for k, v in sd_ckpt.items():
    nk = k
    if nk.startswith('training_module._model.'):
        nk = nk.replace('training_module._model.', '')
    elif nk.startswith('training_module.'):
        nk = nk.replace('training_module.', '')
    sd_stripped[nk] = v

model_sd = scratch_model.state_dict()

only_in_ckpt = [k for k in sd_stripped.keys() if k not in model_sd]
only_in_model = [k for k in model_sd.keys() if k not in sd_stripped]

shape_mismatches = []
matching = []
diff_norms = []

for k, v_ck in sd_stripped.items():
    if k in model_sd:
        v_model = model_sd[k]
        if v_model.shape != v_ck.shape:
            shape_mismatches.append((k, tuple(v_ck.shape), tuple(v_model.shape)))
        else:
            try:
                diff = (v_model.cpu().float() - v_ck.cpu().float()).norm().item()
                diff_norms.append((k, diff))
                matching.append(k)
            except Exception as e:
                matching.append(k)

print('Summary:')
print('  checkpoint params:', len(sd_stripped))
print('  model params:', len(model_sd))
print('  keys only in checkpoint:', len(only_in_ckpt))
print('  keys only in model:', len(only_in_model))
print('  shape mismatches:', len(shape_mismatches))
print('  exact-shape matches:', len(matching))

if only_in_ckpt:
    print('\nSome keys present only in checkpoint (sample 20):')
    for k in only_in_ckpt[:20]:
        print('  ', k)

if only_in_model:
    print('\nSome keys present only in model (sample 20):')
    for k in only_in_model[:20]:
        print('  ', k)

if shape_mismatches:
    print('\nShape mismatches (sample 20):')
    for k, sk, sm in shape_mismatches[:20]:
        print('  ', k, 'ckpt_shape=', sk, 'model_shape=', sm)

if diff_norms:
    diff_norms.sort(key=lambda x: x[1], reverse=True)
    print('\nTop 20 parameter L2 diffs (largest first):')
    for k, d in diff_norms[:20]:
        print('  ', k, f'diff_norm={d:.6f}')

# Save detailed report for later inspection
report = {
    'only_in_ckpt_count': len(only_in_ckpt),
    'only_in_model_count': len(only_in_model),
    'shape_mismatches_count': len(shape_mismatches),
    'matching_count': len(matching),
    'only_in_ckpt_sample': only_in_ckpt[:50],
    'only_in_model_sample': only_in_model[:50],
    'shape_mismatches_sample': shape_mismatches[:50],
    'top_diffs': diff_norms[:50],
}
torch.save(report, 'ckpt_vs_model_report.pt')
print('\nDetailed report saved to ckpt_vs_model_report.pt')


Summary:
  checkpoint params: 147
  model params: 56
  keys only in checkpoint: 133
  keys only in model: 42
  shape mismatches: 7
  exact-shape matches: 7

Some keys present only in checkpoint (sample 20):
   layers.0._arch_alpha
   layers.0.0.1.weight
   layers.0.0.1._arch_alpha.layer1_out_channels
   layers.0.0.2.weight
   layers.0.0.2.bias
   layers.0.0.2.running_mean
   layers.0.0.2.running_var
   layers.0.0.2.num_batches_tracked
   layers.0.0.2._arch_alpha.layer1_out_channels
   layers.0.1.0.weight
   layers.0.1.0._arch_alpha.layer1_out_channels
   layers.0.1.2.weight
   layers.0.1.2.bias
   layers.0.1.2.running_mean
   layers.0.1.2.running_var
   layers.0.1.2.num_batches_tracked
   layers.0.1.2._arch_alpha.layer1_out_channels
   layers.1._arch_alpha
   layers.1.0.1.weight
   layers.1.0.1._arch_alpha.layer1_out_channels

Some keys present only in model (sample 20):
   layers.0.0.weight
   layers.0.2.weight
   layers.0.2.bias
   layers.0.2.running_mean
   layers.0.2.running_var
  

In [87]:
# Update `scratch_model` fully-connected layers to match checkpoint shapes and reload weights
import torch
import DoReFaLayers

ckpt_path = './checkpoints/best-checkpoint.ckpt'
ckpt = torch.load(ckpt_path, map_location='cpu')
sd_ckpt = ckpt.get('state_dict', {})
# strip lightning prefixes
sd_stripped = {}
for k, v in sd_ckpt.items():
    nk = k
    if nk.startswith('training_module._model.'):
        nk = nk.replace('training_module._model.', '')
    elif nk.startswith('training_module.'):
        nk = nk.replace('training_module.', '')
    sd_stripped[nk] = v

# Helper to read weight shapes
def get_shape(key):
    t = sd_stripped.get(key)
    return tuple(t.shape) if t is not None else None

print('Checkpoint fc shapes:')
print(' fc1:', get_shape('fc1.weight'))
print(' fc2:', get_shape('fc2.weight'))
print(' fc3:', get_shape('fc3.weight'))
print(' classifier:', get_shape('classifier.weight'))

# Replace linear layers on scratch_model if sizes found
replaced = []
for name in ['fc1','fc2','fc3','classifier']:
    w_key = f'{name}.weight'
    b_key = f'{name}.bias'
    shape = get_shape(w_key)
    if shape is None:
        continue
    out_f, in_f = shape[0], shape[1]
    print(f'Replacing {name} -> out={out_f}, in={in_f}')
    bias_flag = b_key in sd_stripped
    # Create replacement FrozenDoReFaLinear
    new_layer = DoReFaLayers.FrozenDoReFaLinear(in_features=in_f, out_features=out_f, bias=bias_flag, num_bits=4)
    # copy weights/bias
    new_layer.weight.data.copy_(sd_stripped[w_key].to(new_layer.weight.device))
    if bias_flag:
        new_layer.bias.data.copy_(sd_stripped[b_key].to(new_layer.bias.device))
    setattr(scratch_model, name, new_layer)
    replaced.append(name)

print('Replaced layers:', replaced)

# Try loading remaining checkpoint params
load_report = scratch_model.load_state_dict(sd_stripped, strict=False)
print('Load report after replacement:', load_report)

# Save updated model state for inspection
torch.save(scratch_model.state_dict(), 'scratch_model_post_replace.pt')
print('Saved updated scratch_model state to scratch_model_post_replace.pt')


Checkpoint fc shapes:
 fc1: (128, 198)
 fc2: (128, 128)
 fc3: (64, 128)
 classifier: (10, 64)
Replacing fc1 -> out=128, in=198
Replacing fc2 -> out=128, in=128
Replacing fc3 -> out=64, in=128
Replacing classifier -> out=10, in=64
Replaced layers: ['fc1', 'fc2', 'fc3', 'classifier']
Load report after replacement: _IncompatibleKeys(missing_keys=['layers.0.0.weight', 'layers.0.2.weight', 'layers.0.2.bias', 'layers.0.2.running_mean', 'layers.0.2.running_var', 'layers.1.0.weight', 'layers.1.2.weight', 'layers.1.2.bias', 'layers.1.2.running_mean', 'layers.1.2.running_var', 'layers.2.0.weight', 'layers.2.2.weight', 'layers.2.2.bias', 'layers.2.2.running_mean', 'layers.2.2.running_var', 'layers.3.0.weight', 'layers.3.2.weight', 'layers.3.2.bias', 'layers.3.2.running_mean', 'layers.3.2.running_var', 'layers.4.1.weight', 'layers.4.2.weight', 'layers.4.2.bias', 'layers.4.2.running_mean', 'layers.4.2.running_var', 'layers.5.0.weight', 'layers.5.2.weight', 'layers.5.2.bias', 'layers.5.2.running_mea

## Load weights

In [78]:
scratch_model.load_state_dict(best_arch_state_dict)

<All keys matched successfully>

In [79]:
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total_samples
    return avg_loss, accuracy

In [80]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

scratch_model.train()
module = DartsClassificationModule(1e-5, 0., 0., max_epochs)
module.set_model(scratch_model)
trainer = Trainer(max_epochs=module.max_epochs, precision=16, gradient_clip_val=0.1)

/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/lightning_fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [88]:
trainer.fit(module, train_loader)

/home/filippo/miniconda3/envs/pytorch-env/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/filippo/TESI_FILIPPO_LUCCHESI/lightning_logs/version_1/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type             | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | criterion | CrossEntropyLoss | 0      | train | 0    
1 | metrics   | ModuleDict       | 0      | train | 0    
2 | _model    | CustomDARTSSpace | 114 K  | eval  | 0    
---------------------------------------------------------------
114 K     Trainable params
0         Non-trainable params
114 K     Total params
0.460     Total estimated model params size (MB)
7         Modules in train mode
42        Modules in eval mode
0         Total Flops


`Trainer.fit` stopped: `max_epochs=200` reached.


KeyError: 'train_acc'

In [90]:
criterion = nn.CrossEntropyLoss()
scratch_model.to(device)

avg_loss, accuracy = evaluate_model(scratch_model, valid_loader, criterion, device)
print(f'Validation Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4%}')

Validation Loss: 3.0711, Accuracy: 10.1700%


In [83]:
for inputs, labels in valid_loader:
    inputs, labels = inputs.to(device), labels.to(device)
    outputs = scratch_model(inputs)
    print('Predicted:', outputs.argmax(dim=1))
    print('True:', labels)
    break

Predicted: tensor([9, 5, 7, 8, 8, 5, 7, 9, 9, 8, 7, 5, 7, 7, 7, 8, 8, 9, 5, 9, 7, 9, 9, 8,
        7, 9, 9, 8, 9, 9, 8, 9, 7, 9, 5, 8, 9, 8, 2, 8, 8, 7, 8, 5, 7, 7, 7, 9,
        7, 8, 9, 9, 7, 7, 9, 5, 8, 7, 8, 8, 9, 9, 7, 7, 5, 7, 7, 2, 9, 8, 5, 8,
        8, 8, 7, 7, 8, 7, 9, 8, 7, 8, 2, 5, 8, 8, 7, 8, 5, 8, 8, 7, 9, 5, 8, 9,
        7, 7, 8, 9, 5, 8, 8, 8, 7, 9, 8, 9, 7, 9, 8, 8, 7, 7, 9, 8, 7, 7, 8, 9,
        7, 7, 7, 9, 7, 9, 7, 8], device='cuda:0')
True: tensor([3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 0, 4, 9,
        5, 2, 4, 0, 9, 6, 6, 5, 4, 5, 9, 2, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 3, 9,
        7, 6, 9, 8, 0, 3, 8, 8, 7, 7, 4, 6, 7, 3, 6, 3, 6, 2, 1, 2, 3, 7, 2, 6,
        8, 8, 0, 2, 9, 3, 3, 8, 8, 1, 1, 7, 2, 5, 2, 7, 8, 9, 0, 3, 8, 6, 4, 6,
        6, 0, 0, 7, 4, 5, 6, 3, 1, 1, 3, 6, 8, 7, 4, 0, 6, 2, 1, 3, 0, 4, 2, 7,
        8, 3, 1, 2, 8, 0, 8, 3], device='cuda:0')


In [89]:
# Forward-only epoch to refresh BatchNorm running stats
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

scratch_model.to(device)
scratch_model.train()

n_batches = 0
with torch.no_grad():
    for i, (inputs, labels) in enumerate(train_loader):
        inputs = inputs.to(device)
        _ = scratch_model(inputs)
        n_batches += 1
print(f'Completed forward-only pass over {n_batches} batches to update BN running stats')
# Save model state after BN update for inspection
torch.save(scratch_model.state_dict(), 'scratch_model_bn_updated.pt')
print('Saved BN-updated state to scratch_model_bn_updated.pt')


Device: cuda
Completed forward-only pass over 391 batches to update BN running stats
Saved BN-updated state to scratch_model_bn_updated.pt


In [91]:
# Diagnostics: explain DARTS high acc vs poor standalone accuracy
import torch
import pprint

print('--- Optimizer/LR context ---')
print('search evaluator LR (expected):', getattr(evaluator.module, 'learning_rate', 'N/A') if 'evaluator' in globals() else 'evaluator missing')
print('standalone module LR (if present):', getattr(module, 'learning_rate', 'module missing') if 'module' in globals() else 'module missing')

print('\n--- Data transform context ---')
if 'train_data' in globals() and hasattr(train_data, 'transform'):
    print('train transform:', train_data.transform)
if 'test_data' in globals() and hasattr(test_data, 'transform'):
    print('test transform:', test_data.transform)

print('\n--- Checkpoint alignment context ---')
ckpt = torch.load('./checkpoints/best-checkpoint.ckpt', map_location='cpu')
sd = ckpt.get('state_dict', {})
sd_stripped = {}
for k, v in sd.items():
    nk = k
    if nk.startswith('training_module._model.'):
        nk = nk.replace('training_module._model.', '')
    elif nk.startswith('training_module.'):
        nk = nk.replace('training_module.', '')
    sd_stripped[nk] = v

msd = scratch_model.state_dict()
common = [k for k in msd if k in sd_stripped and msd[k].shape == sd_stripped[k].shape]
mismatch = [k for k in msd if k in sd_stripped and msd[k].shape != sd_stripped[k].shape]
only_model = [k for k in msd if k not in sd_stripped]
only_ckpt = [k for k in sd_stripped if k not in msd]
print('model keys:', len(msd))
print('ckpt keys (stripped):', len(sd_stripped))
print('common same-shape keys:', len(common))
print('shape mismatches:', len(mismatch))
print('only model:', len(only_model))
print('only checkpoint:', len(only_ckpt))

if len(common) > 0:
    # Average relative norm difference on common tensors: lower is better; near 0 means highly aligned
    rel_diffs = []
    for k in common:
        a = msd[k].float().cpu()
        b = sd_stripped[k].float().cpu()
        denom = b.norm().item() + 1e-12
        rel_diffs.append((a - b).norm().item() / denom)
    print('mean relative tensor diff (common keys):', float(sum(rel_diffs) / len(rel_diffs)))

print('\n--- Quick train-batch sanity ---')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
scratch_model.to(device).eval()
with torch.no_grad():
    x, y = next(iter(train_loader))
    x, y = x.to(device), y.to(device)
    out = scratch_model(x)
    acc1 = (out.argmax(dim=1) == y).float().mean().item()
print('single train-batch top1:', acc1)

--- Optimizer/LR context ---
search evaluator LR (expected): 0.01
standalone module LR (if present): 1e-05

--- Data transform context ---
train transform: Compose(
    RandomCrop(size=(32, 32), padding=4)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-15.0, 15.0], interpolation=nearest, expand=False, fill=0)
    ToTensor()
)
test transform: Compose(
    ToTensor()
)

--- Checkpoint alignment context ---
model keys: 56
ckpt keys (stripped): 147
common same-shape keys: 14
shape mismatches: 0
only model: 42
only checkpoint: 133
mean relative tensor diff (common keys): 27928571428571.53

--- Quick train-batch sanity ---
single train-batch top1: 0.109375


In [92]:
# Strict checkpoint compatibility gate for current scratch_model (must be >=95% loadable keys)
import torch
from collections import OrderedDict

def strip_lightning_prefixes(state_dict):
    out = OrderedDict()
    for k, v in state_dict.items():
        nk = k
        if nk.startswith('training_module._model.'):
            nk = nk.replace('training_module._model.', '')
        elif nk.startswith('training_module.'):
            nk = nk.replace('training_module.', '')
        out[nk] = v
    return out

ckpt = torch.load('./checkpoints/best-checkpoint.ckpt', map_location='cpu')
sd_strict = strip_lightning_prefixes(ckpt['state_dict'])
msd = scratch_model.state_dict()

loadable = [k for k in msd if (k in sd_strict and msd[k].shape == sd_strict[k].shape)]
coverage = len(loadable) / max(len(msd), 1)
print(f'Scratch-model strict-load coverage: {len(loadable)}/{len(msd)} = {coverage:.2%}')

if coverage < 0.95:
    raise RuntimeError(
        f'STRICT GATE FAILED: only {coverage:.2%} of scratch_model keys are strictly loadable from checkpoint (<95%). '
        'Do not trust scratch_model metrics; use exact checkpoint model path below.'
    )
print('STRICT GATE PASSED')

Scratch-model strict-load coverage: 14/56 = 25.00%


RuntimeError: STRICT GATE FAILED: only 25.00% of scratch_model keys are strictly loadable from checkpoint (<95%). Do not trust scratch_model metrics; use exact checkpoint model path below.

In [97]:
# Exact-checkpoint fallback path: load strictly compatible tensors, report coverage, and evaluate
import torch
from collections import OrderedDict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint_path = './checkpoints/best-checkpoint.ckpt'
ckpt = torch.load(checkpoint_path, map_location='cpu')
raw_sd = ckpt.get('state_dict', {})

model_sd = OrderedDict()
for k, v in raw_sd.items():
    if k.startswith('training_module._model.'):
        model_sd[k.replace('training_module._model.', '')] = v

exact_model = CustomDARTSSpace(input_channels=3, channels=64, num_classes=10, layers=7, verbose=0, num_bits=4)
target_sd = exact_model.state_dict()

compatible = OrderedDict()
for k, v in model_sd.items():
    if k in target_sd and target_sd[k].shape == v.shape:
        compatible[k] = v

coverage = len(compatible) / max(len(target_sd), 1)
print(f'Compatible load coverage into runtime model: {len(compatible)}/{len(target_sd)} = {coverage:.2%}')

load_info = exact_model.load_state_dict(compatible, strict=False)
print('Compatible-load report:', load_info)

exact_model.to(device).eval()
criterion = nn.CrossEntropyLoss()
pre_loss, pre_acc = evaluate_model(exact_model, valid_loader, criterion, device)
print(f'Pre-finetune eval -> loss: {pre_loss:.4f}, acc: {pre_acc:.4%}')

Compatible load coverage into runtime model: 29/98 = 29.59%
Compatible-load report: _IncompatibleKeys(missing_keys=['layers.0.0.1.weight', 'layers.0.0.2.weight', 'layers.0.0.2.bias', 'layers.0.0.2.running_mean', 'layers.0.0.2.running_var', 'layers.0.1.0.weight', 'layers.0.1.2.weight', 'layers.0.1.2.bias', 'layers.0.1.2.running_mean', 'layers.0.1.2.running_var', 'layers.1.0.1.weight', 'layers.1.0.2.weight', 'layers.1.0.2.bias', 'layers.1.0.2.running_mean', 'layers.1.0.2.running_var', 'layers.1.1.0.weight', 'layers.1.1.2.weight', 'layers.1.1.2.bias', 'layers.1.1.2.running_mean', 'layers.1.1.2.running_var', 'layers.2.0.1.weight', 'layers.2.0.2.weight', 'layers.2.0.2.bias', 'layers.2.0.2.running_mean', 'layers.2.0.2.running_var', 'layers.2.1.0.weight', 'layers.2.1.2.weight', 'layers.2.1.2.bias', 'layers.2.1.2.running_mean', 'layers.2.1.2.running_var', 'layers.3.0.1.weight', 'layers.3.0.2.weight', 'layers.3.0.2.bias', 'layers.3.0.2.running_mean', 'layers.3.0.2.running_var', 'layers.3.1.0.we

Pre-finetune eval -> loss: 4.3540, acc: 9.1800%


In [98]:
# One-epoch fine-tune on exact checkpoint model with sensible LR
import torch

exact_model.train()
optimizer = torch.optim.SGD(exact_model.parameters(), lr=1e-3, momentum=0.9, weight_decay=0.0)
criterion = nn.CrossEntropyLoss()

total = 0
correct = 0
running_loss = 0.0

for inputs, labels in train_loader:
    inputs, labels = inputs.to(device), labels.to(device)
    optimizer.zero_grad(set_to_none=True)
    outputs = exact_model(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    running_loss += loss.item() * labels.size(0)
    correct += (outputs.argmax(dim=1) == labels).sum().item()
    total += labels.size(0)

train_loss_epoch = running_loss / max(total, 1)
train_acc_epoch = correct / max(total, 1)
print(f'Fine-tune epoch (lr=1e-3) -> train_loss: {train_loss_epoch:.4f}, train_acc: {train_acc_epoch:.4%}')

Fine-tune epoch (lr=1e-3) -> train_loss: 2.1005, train_acc: 20.8880%


In [99]:
# Re-evaluate after one epoch fine-tuning and save weights
exact_model.eval()
post_loss, post_acc = evaluate_model(exact_model, valid_loader, criterion, device)
print(f'Post fine-tune eval -> loss: {post_loss:.4f}, acc: {post_acc:.4%}')

torch.save(exact_model.state_dict(), 'exact_checkpoint_model_after_1epoch_lr1e3.pt')
print('Saved weights to exact_checkpoint_model_after_1epoch_lr1e3.pt')

Post fine-tune eval -> loss: 1.9532, acc: 24.7300%
Saved weights to exact_checkpoint_model_after_1epoch_lr1e3.pt


## Deterministic Architecture Retrain (Clean Baseline)

In [103]:
# Build a deterministic model from exported architecture choices, then reinitialize for clean retraining
import copy
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

det_model = best_arch.freeze(best_arch_desc)
det_model = build_frozen_dorefa_model(det_model, num_bits=4)

# Fresh random init to create a true from-scratch baseline using the chosen architecture only
for m in det_model.modules():
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight, a=5 ** 0.5)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.ones_(m.weight)
        nn.init.zeros_(m.bias)

det_model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer_det = torch.optim.SGD(det_model.parameters(), lr=1e-2, momentum=0.9, weight_decay=0.0)
scheduler_det = torch.optim.lr_scheduler.StepLR(optimizer_det, step_size=15, gamma=0.5)

det_pre_loss, det_pre_acc = evaluate_model(det_model, valid_loader, criterion, device)
print(f'Deterministic scratch model pre-train -> val_loss: {det_pre_loss:.4f}, val_acc: {det_pre_acc:.4%}')

Deterministic scratch model pre-train -> val_loss: 4.8678, val_acc: 9.9700%


In [104]:
# Train deterministic model for a few epochs with same optimizer/scheduler settings as DARTS
det_epochs = max_epochs
history = []

for epoch in range(1, det_epochs + 1):
    det_model.train()
    total = 0
    correct = 0
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer_det.zero_grad(set_to_none=True)
        outputs = det_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_det.step()

        running_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / max(total, 1)
    train_acc = correct / max(total, 1)
    val_loss, val_acc = evaluate_model(det_model, valid_loader, criterion, device)
    history.append((epoch, train_loss, train_acc, val_loss, val_acc, optimizer_det.param_groups[0]['lr']))

    print(
        f"Epoch {epoch}/{det_epochs} | lr={optimizer_det.param_groups[0]['lr']:.5f} "
        f"| train_loss={train_loss:.4f} train_acc={train_acc:.4%} "
        f"| val_loss={val_loss:.4f} val_acc={val_acc:.4%}"
    )

    scheduler_det.step()

torch.save(det_model.state_dict(), 'det_model_scratch_darts_opt_3epochs.pt')
print('Saved deterministic retrained weights to det_model_scratch_darts_opt_3epochs.pt')

Epoch 1/200 | lr=0.01000 | train_loss=2.2318 train_acc=15.0040% | val_loss=2.1845 val_acc=15.5200%
Epoch 2/200 | lr=0.01000 | train_loss=2.1866 train_acc=15.6380% | val_loss=2.1898 val_acc=15.8100%
Epoch 3/200 | lr=0.01000 | train_loss=2.1426 train_acc=16.6920% | val_loss=2.1022 val_acc=15.5300%
Epoch 4/200 | lr=0.01000 | train_loss=2.1113 train_acc=16.9200% | val_loss=2.0971 val_acc=17.1000%
Epoch 5/200 | lr=0.01000 | train_loss=2.1207 train_acc=16.6620% | val_loss=2.1109 val_acc=16.5200%
Epoch 6/200 | lr=0.01000 | train_loss=2.0830 train_acc=16.7200% | val_loss=2.0389 val_acc=18.1400%
Epoch 7/200 | lr=0.01000 | train_loss=2.0575 train_acc=16.9880% | val_loss=2.1026 val_acc=15.5600%
Epoch 8/200 | lr=0.01000 | train_loss=2.0469 train_acc=17.5640% | val_loss=1.9995 val_acc=17.9600%
Epoch 9/200 | lr=0.01000 | train_loss=2.0053 train_acc=19.0880% | val_loss=1.9690 val_acc=17.9800%
Epoch 10/200 | lr=0.01000 | train_loss=1.9916 train_acc=19.3760% | val_loss=2.0815 val_acc=19.2200%
Epoch 11/

In [102]:
# Diagnostic: compare checkpoint arch-alpha selections to best_arch_desc
import torch
from collections import defaultdict

ckpt = torch.load('./checkpoints/best-checkpoint.ckpt', map_location='cpu')
sd = ckpt.get('state_dict', {})

# Known choice option lists from CustomDARTSSpace definition
choice_options = {
    'layer1_out_channels': [16, 32, 64],
    'layer2_out_channels': [16, 32, 64],
    'layer3_out_channels': [16, 32, 64],
    'layer4_out_channels': [16, 32, 64],
    'layer5_out_channels': [16, 32, 64],
    'layer6_out_channels': [16, 32, 64],
    'feature1': [32, 64, 128],
    'feature2': [32, 64, 128],
    'feature3': [32, 64],
}

# Extract arch_alpha tensors and their argmax selections
arch_alpha_info = []
for k, v in sd.items():
    if '._arch_alpha.' in k:
        # Example key: training_module._model.layers.0.0.1._arch_alpha.layer1_out_channels
        prefix, choice_name = k.split('._arch_alpha.')
        prefix = prefix.replace('training_module._model.', '')
        tensor = v
        if isinstance(tensor, torch.Tensor):
            idx = int(tensor.argmax().item())
            mapped = choice_options.get(choice_name)
            mapped_val = mapped[idx] if mapped and idx < len(mapped) else None
            arch_alpha_info.append((prefix, choice_name, idx, mapped_val, tensor.shape))

# Print discovered choices from checkpoint
print('Discovered arch_alpha choices from checkpoint:')
for prefix, choice_name, idx, mapped_val, shape in arch_alpha_info:
    print(f'  location={prefix:40s} choice={choice_name:25s} argmax_idx={idx} mapped_val={mapped_val} tensor_shape={shape}')

# Compare with best_arch_desc
print('\nBest architecture description (best_arch_desc):')
try:
    from pprint import pprint
    pprint(best_arch_desc)
except Exception:
    print('best_arch_desc not available in this kernel')

# Compare per-choice agreement
agree = 0
total = 0
for _prefix, choice_name, idx, mapped_val, _shape in arch_alpha_info:
    total += 1
    chosen_val = None
    if isinstance(best_arch_desc, dict) and choice_name in best_arch_desc:
        chosen_val = best_arch_desc[choice_name]
    if mapped_val is not None and chosen_val is not None:
        ok = (mapped_val == chosen_val)
    else:
        ok = False
    print(f"choice {choice_name:25s}: checkpoint_sel={mapped_val} best_arch_desc={chosen_val} MATCH={ok}")
    if ok:
        agree += 1

print(f'\nChoice agreement: {agree}/{total} = {agree/total:.2%}' if total>0 else 'No arch_alpha choices found in checkpoint')

# Also compare classifier/FC shapes between checkpoint and built models
print('\nSample layer shapes:')
# checkpoint fc shapes
for name in ['fc1.weight','fc2.weight','fc3.weight','classifier.weight']:
    if f'training_module._model.{name}' in sd:
        print('  ckpt', name, tuple(sd[f'training_module._model.{name}'].shape))
    elif name in sd:
        print('  ckpt', name, tuple(sd[name].shape))
# runtime scratch_model shapes
if 'scratch_model' in globals():
    ssd = scratch_model.state_dict()
    for name in ['fc1.weight','fc2.weight','fc3.weight','classifier.weight']:
        if name in ssd:
            print('  scratch_model', name, tuple(ssd[name].shape))

print('\nIf many choices disagree, the architecture used during checkpoint training differs from the exported best_arch_desc/frozen model.')


Discovered arch_alpha choices from checkpoint:
  location=layers.0.0.1                             choice=layer1_out_channels       argmax_idx=0 mapped_val=16 tensor_shape=torch.Size([3])
  location=layers.0.0.2                             choice=layer1_out_channels       argmax_idx=0 mapped_val=16 tensor_shape=torch.Size([3])
  location=layers.0.1.0                             choice=layer1_out_channels       argmax_idx=0 mapped_val=16 tensor_shape=torch.Size([3])
  location=layers.0.1.2                             choice=layer1_out_channels       argmax_idx=0 mapped_val=16 tensor_shape=torch.Size([3])
  location=layers.1.0.1                             choice=layer1_out_channels       argmax_idx=0 mapped_val=16 tensor_shape=torch.Size([3])
  location=layers.1.0.1                             choice=layer2_out_channels       argmax_idx=0 mapped_val=16 tensor_shape=torch.Size([3])
  location=layers.1.0.2                             choice=layer2_out_channels       argmax_idx=0 mapped_va

In [108]:
# Reconstruct the checkpoint-matched architecture (weights ignored; architecture must match exactly)
import json
import re
import torch
import torch.nn as nn
from collections import Counter, defaultdict

choice_options = {
    'layer1_out_channels': [16, 32, 64],
    'layer2_out_channels': [16, 32, 64],
    'layer3_out_channels': [16, 32, 64],
    'layer4_out_channels': [16, 32, 64],
    'layer5_out_channels': [16, 32, 64],
    'layer6_out_channels': [16, 32, 64],
    'feature1': [32, 64, 128],
    'feature2': [32, 64, 128],
    'feature3': [32, 64],
}

def derive_checkpoint_arch_desc(checkpoint_path: str):
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    state_dict = ckpt.get('state_dict', {})
    votes = defaultdict(list)

    for key, tensor in state_dict.items():
        if not isinstance(tensor, torch.Tensor) or '._arch_alpha' not in key:
            continue

        match = re.search(r'\blayers\.(\d+)\._arch_alpha$', key)
        if match:
            choice_name = f'layer_{int(match.group(1)) + 1}'
        else:
            if '._arch_alpha.' not in key:
                continue
            choice_name = key.split('._arch_alpha.', 1)[1]
        idx = int(tensor.argmax().item())
        votes[choice_name].append(idx)

    arch_desc = {}
    for choice_name, idxs in votes.items():
        idx = Counter(idxs).most_common(1)[0][0]
        options = choice_options.get(choice_name)
        arch_desc[choice_name] = options[idx] if options is not None else idx

    return arch_desc, votes

class ExactDARTSArchitecture(nn.Module):
    def __init__(self, arch_desc):
        super().__init__()
        self.arch_desc = arch_desc
        self.preliminary_layer = nn.Conv2d(3, 16, kernel_size=3, padding=0, bias=False)
        self.layer0_bn = nn.BatchNorm2d(16)
        self.layer0_relu = nn.ReLU(inplace=False)

        layer_out_channels = [
            arch_desc['layer1_out_channels'],
            arch_desc['layer2_out_channels'],
            arch_desc['layer3_out_channels'],
            arch_desc['layer4_out_channels'],
            arch_desc['layer5_out_channels'],
            arch_desc['layer6_out_channels'],
            22,
        ]

        self.layers = nn.ModuleList()
        in_channels = 16
        for layer_index, out_channels in enumerate(layer_out_channels, start=1):
            if arch_desc[f'layer_{layer_index}'] == 0:
                block = nn.Sequential(
                    nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                    nn.Conv2d(in_channels, out_channels, kernel_size=3, bias=False),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU(inplace=False),
                )
            else:
                block = nn.Sequential(
                    nn.Conv2d(in_channels, out_channels, kernel_size=3, bias=False),
                    nn.AvgPool2d(kernel_size=2, stride=1, padding=1),
                    nn.BatchNorm2d(out_channels),
                    nn.ReLU(inplace=False),
                )
            self.layers.append(block)
            in_channels = out_channels

        self.pool = nn.AdaptiveAvgPool2d((3, 3))
        self.fc1 = nn.Linear(198, arch_desc['feature1'])
        self.fc2 = nn.Linear(arch_desc['feature1'], arch_desc['feature2'])
        self.fc3 = nn.Linear(arch_desc['feature2'], arch_desc['feature3'])
        self.relu = nn.ReLU(inplace=False)
        self.classifier = nn.Linear(arch_desc['feature3'], 10)

    def forward(self, x):
        x = self.preliminary_layer(x)
        x = self.layer0_bn(x)
        x = self.layer0_relu(x)

        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i in (1, 3, 6):
                x = nn.AvgPool2d(kernel_size=2, stride=2)(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.classifier(x)
        return x

checkpoint_arch_desc, checkpoint_votes = derive_checkpoint_arch_desc('./checkpoints/best-checkpoint.ckpt')

print('Checkpoint-derived architecture choices:')
for name in sorted(checkpoint_arch_desc):
    print(f'  {name}: {checkpoint_arch_desc[name]}   votes={dict(Counter(checkpoint_votes[name]))}')

# Compare directly against the exported architecture from the search run.
if isinstance(best_arch_desc, dict):
    all_keys = sorted(set(checkpoint_arch_desc) | set(best_arch_desc))
    diffs = []
    for key in all_keys:
        if checkpoint_arch_desc.get(key) != best_arch_desc.get(key):
            diffs.append((key, checkpoint_arch_desc.get(key), best_arch_desc.get(key)))
    print(f'\nArchitecture dict equality with best_arch_desc: {len(diffs) == 0}')
    if diffs:
        print('Differing choice values:')
        for key, ckpt_value, best_value in diffs:
            print(f'  {key}: checkpoint={ckpt_value} best_arch_desc={best_value}')

conflicts = {name: sorted(set(idxs)) for name, idxs in checkpoint_votes.items() if len(set(idxs)) > 1}
if conflicts:
    print('\nConflicting argmax votes found for some choices:')
    for name, idxs in conflicts.items():
        print(f'  {name}: {idxs}')
else:
    print('\nNo conflicting argmax votes found.')

# Build a plain deterministic architecture from the checkpoint-derived choices.
reconstructed_arch = ExactDARTSArchitecture(checkpoint_arch_desc)
reconstructed_arch = build_frozen_dorefa_model(reconstructed_arch, num_bits=4)

# Save the exact architecture description for reproducibility.
with open('checkpoint_arch_desc.json', 'w') as f:
    json.dump(checkpoint_arch_desc, f, indent=2)
print('\nSaved checkpoint architecture description to checkpoint_arch_desc.json')

# Make this the active model for later retraining cells.
scratch_model = reconstructed_arch
print('Active model set to checkpoint-derived reconstruction:', type(scratch_model).__name__)

Checkpoint-derived architecture choices:
  feature1: 32   votes={0: 2}
  feature2: 32   votes={0: 2}
  feature3: 32   votes={0: 2}
  layer1_out_channels: 16   votes={0: 6}
  layer2_out_channels: 16   votes={0: 6}
  layer3_out_channels: 64   votes={2: 6}
  layer4_out_channels: 32   votes={1: 6}
  layer5_out_channels: 64   votes={2: 6}
  layer6_out_channels: 16   votes={0: 6}
  layer_1: 1   votes={1: 1}
  layer_2: 1   votes={1: 1}
  layer_3: 1   votes={1: 1}
  layer_4: 1   votes={1: 1}
  layer_5: 0   votes={0: 1}
  layer_6: 1   votes={1: 1}
  layer_7: 1   votes={1: 1}

Architecture dict equality with best_arch_desc: True

No conflicting argmax votes found.



Saved checkpoint architecture description to checkpoint_arch_desc.json
Active model set to checkpoint-derived reconstruction: ExactDARTSArchitecture


In [110]:
def evaluate_model(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total_samples += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total_samples
    return avg_loss, accuracy
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

scratch_model.train()
module = DartsClassificationModule(1e-5, 0., 0., max_epochs)
module.set_model(scratch_model)
trainer = Trainer(max_epochs=3, precision=16, gradient_clip_val=0.1)
trainer.fit(module, train_loader)
criterion = nn.CrossEntropyLoss()
scratch_model.to(device)

avg_loss, accuracy = evaluate_model(scratch_model, valid_loader, criterion, device)
print(f'Validation Loss: {avg_loss:.4f}, Accuracy: {accuracy:.4%}')
for inputs, labels in valid_loader:
    inputs, labels = inputs.to(device), labels.to(device)
    outputs = scratch_model(inputs)
    print('Predicted:', outputs.argmax(dim=1))
    print('True:', labels)
    break

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type                   | Params | Mode  | FLOPs
---------------------------------------------------------------------
0 | criterion | CrossEntropyLoss       | 0      | train | 0    
1 | metrics   | ModuleDict             | 0      | train | 0    
2 | _model    | ExactDARTSArchitecture | 72.8 K | train | 0    
---------------------------------------------------------------------
72

Epoch 2: 100%|███| 391/391 [00:10<00:00, 38.14it/s, v_num=3, train_loss=3.640, train_acc=0.125]

`Trainer.fit` stopped: `max_epochs=3` reached.


Epoch 2: 100%|███| 391/391 [00:10<00:00, 38.10it/s, v_num=3, train_loss=3.640, train_acc=0.125]
[2026-04-30 16:50:08] Final result: 0.125


RuntimeError: DataLoader worker (pid(s) 1487871, 1487872, 1487873, 1487874, 1487875, 1487876) exited unexpectedly